#  03) Find Gaia stars in all DDF

- creation date : 2026-04-10
- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings
from astroquery.gaia import Gaia

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "GAIA_STARCAT_DR3_03"
INDIR_DATA = "data_GAIA_STARCAT_DR3_02"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

### DDF definition

In [ ]:
CONE_RADIUS = 1800.0  # Cone search radius in arcsec (0.5 deg per DDF)

BANDS = list("ugrizy")

# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

## Read all Gaia stars from  Deep field

In [ ]:
# Boucle sur les champs profonds

all_df = []
for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f"\nTraitement du champ {field_name} (RA={ra}, Dec={dec})...")

    try:
        # Sauvegarder les résultats dans un fichier CSV
        input_filename = f"{INDIR_DATA}/allgaia_sources_{field_name}.csv"
        df = pd.read_csv(input_filename, index_col=0)
        df["DDF"] = field_name
        all_df.append(df)

    except Exception as e:
        print(f"Erreur lors du traitement du champ {field_name}: {e}")

## Concatenate and set a STABLE Flag

In [ ]:
df = pd.concat(all_df)

In [ ]:
df

In [ ]:
stable_mask = (df["in_vari_classification_result"].fillna(0) == 0) & (
    df["in_vari_eclipsing_binary"].fillna(0) == 0
)

In [ ]:
df["STABLE"] = stable_mask

In [ ]:
df

## Save all Gaia in a file

In [ ]:
output_filename = f"{DIR_DATA}/allgaia_sources_allddf.csv"
df.to_csv(output_filename)